In [0]:
dbutils.widgets.removeAll()

In [0]:
import json

test_payload = {
        "FileIds": [
            {
            "DataGroupTrackingID": "TRACK_MEMBER_834_02", 
            "DataGroupMappingId": "834MEMBER",
            "FileId": "03",
            "FileLayoutID": "834",
            "FileLayoutDescription": "Standard834",
            "CurrentContainer": "/Volumes/claimsprocessing/bronze/member",
            "CurrentFolderPath": "",
            "ConsolidatedMappingFilePath": "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimMember/Bronze/Schema/Consolidation",
            "ConsolidatedMappingFileName": "ConsolidationMember.json",
            "ConsolidatedLayerDataModelFilePath": "/Workspace/Repos/logi@openhealthagents.org/alphaesai/ClaimsProcessing/DimMember/Bronze/Schema/Consolidation/DataModels",
            "ConsolidatedLayerDataModel": "MemberDataModel.json",
            "ConsolidatedFolderPath": "/Volumes/claimsprocessing/bronze/member_consolidated"
            }
        ]
    }

json_string_input = json.dumps(test_payload)


In [0]:
import json
from pyspark.sql.functions import explode, col
import pydantic

dbutils.widgets.text("ConsolidationJSON", json_string_input)
ConsolidationJSON = dbutils.widgets.get("ConsolidationJSON")

print(f"Received ConsolidationJSON parameter (length: {len(ConsolidationJSON)})")

In [0]:
# Databricks notebook source
import json
from pyspark.sql.functions import explode, col
import pydantic

dbutils.widgets.text("ConsolidationJSON", "", "")
ConsolidationJSON = dbutils.widgets.get("ConsolidationJSON")

print(f"Received ConsolidationJSON parameter (length: {len(ConsolidationJSON)} chars)")

# COMMAND ----------

# Load and explode input JSON parameters (serverless-compatible)
print("Step 1: Parsing JSON and creating DataFrame...")
parsed_json = json.loads(ConsolidationJSON)
file_id_df = spark.createDataFrame([parsed_json])

print("Step 2: Exploding FileIds array...")
exploded_file_ids = file_id_df.select(explode(col("FileIds")).alias("col")).select(
    col("col.DataGroupTrackingID"),
    col("col.DataGroupMappingId"),
    col("col.FileId"),
    col("col.FileLayoutID"),
    col("col.FileLayoutDescription"),
    col("col.CurrentContainer"),
    col("col.CurrentFolderPath"),
    col("col.ConsolidatedMappingFilePath"),
    col("col.ConsolidatedMappingFileName"),
    col("col.ConsolidatedLayerDataModelFilePath"),
    col("col.ConsolidatedLayerDataModel"),
    col("col.ConsolidatedFolderPath")
)

file_count = exploded_file_ids.count()
print(f"Found {file_count} file(s) to process")

In [0]:
# Import helper functions and process_consolidation function
# (Serverless doesn't support %run reliably, so we define inline)

from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import DataFrame, Row
from delta.tables import *
import json

def get_sql_type(data_type):
    """Convert DataModel data type to SQL type string"""
    type_mapping = {
        "StringType": "STRING",
        "IntegerType": "INT",
        "LongType": "BIGINT",
        "DoubleType": "DOUBLE",
        "FloatType": "FLOAT",
        "BooleanType": "BOOLEAN",
        "DateType": "DATE",
        "TimestampType": "TIMESTAMP",
        "DecimalType": "DECIMAL(38,10)"
    }
    return type_mapping.get(data_type, "STRING")

def get_data_type(data_type):
    """Convert DataModel data type to PySpark DataType"""
    type_mapping = {
        "StringType": StringType(),
        "IntegerType": IntegerType(),
        "LongType": LongType(),
        "DoubleType": DoubleType(),
        "FloatType": FloatType(),
        "BooleanType": BooleanType(),
        "DateType": DateType(),
        "TimestampType": TimestampType(),
        "DecimalType": DecimalType(38, 10)
    }
    return type_mapping.get(data_type, StringType())

def get_struct(data_model_df):
    """Create StructType schema from DataModel DataFrame"""
    fields = []
    for row in data_model_df.orderBy("Ordinal").collect():
        field_name = row.FieldName
        data_type = get_data_type(row.DataType)
        fields.append(StructField(field_name, data_type, nullable=True))
    return StructType(fields)

def get_select_expr(cols_df):
    """
    Build SQL select expressions with type casting
    Returns: List of select expression strings
    """
    sql_commands = []
    
    for row in cols_df.collect():
        field_name = row.FieldName
        data_type = row.DataType
        ordinal = row.Ordinal
        # Clean source column name - strip any extraneous quotes
        source_column = row.SourceColumn.strip().strip("'").strip('"') if row.SourceColumn else ""
        destination_column = row.DestinationColumn
        source_column_format = row.SourceColumnFormat if row.SourceColumnFormat else ""
        column_query_raw = row.ColumnQuery if row.ColumnQuery else ""
        column_query = column_query_raw.strip().strip("'").strip('"') if column_query_raw else ""
        
        if column_query and column_query.strip():
            expr = f"NULLIF(CAST({column_query} AS {get_sql_type(data_type)}),'') AS {destination_column}"
        elif data_type == "DateType" and source_column_format:
            expr = f"try_to_date({source_column},'{source_column_format}') AS {destination_column}"
        elif data_type == "DateType":
            # No format specified - try common formats
            expr = f"COALESCE(try_to_date({source_column},'MM/dd/yyyy'), try_to_date({source_column},'yyyy-MM-dd'), try_to_date({source_column},'M/d/yyyy')) AS {destination_column}"
        elif data_type == "TimestampType" and source_column_format:
            expr = f"try_to_timestamp({source_column},'{source_column_format}') AS {destination_column}"
        elif data_type == "TimestampType":
            expr = f"try_to_timestamp({source_column}) AS {destination_column}"
        else:
            expr = f"CAST({source_column} AS {get_sql_type(data_type)}) AS {destination_column}"
        
        sql_commands.append(expr)
    
    return sql_commands

def custom_select(available_cols, required_cols):
    """Add missing columns as null"""
    return [col(column) if column in available_cols else lit(None).alias(column) for column in required_cols]

def process_consolidation(FileId, CurrentContainer, CurrentFolderPath,
                         ConsolidatedLayerDataModel, ConsolidatedLayerDataModelFilePath,
                         ConsolidatedMappingFileName, ConsolidatedMappingFilePath,
                         ConsolidatedFolderPath):
    """
    Process a single file consolidation
    Returns: JSON string with processing result
    """
    rJSON = {}
    
    try:
        # Get notebook context
        try:
            ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            current_job_id = ctx.tags().get("jobId").getOrElse(lambda: "Undefined")
        except Exception:
            current_job_id = "Undefined"
        
        rJSON["CurrentJobId"] = current_job_id
        
        # Construct paths or table names
        # Check if CurrentContainer is a catalog.schema format
        if CurrentContainer.count('.') >= 1 and not CurrentContainer.startswith('/'):
            # It's a catalog.schema format, treat as table
            FullProcessed = f"{CurrentContainer}.{CurrentFolderPath}"
            is_input_table = True
        else:
            # It's a file path
            FullProcessed = f"{CurrentContainer}/{CurrentFolderPath}/"
            is_input_table = False
            
        FullConsolidatedFolderPath = ConsolidatedFolderPath
        DataModelFile = f"{ConsolidatedLayerDataModelFilePath}/{ConsolidatedLayerDataModel}"
        ConsolidationMapping = f"{ConsolidatedMappingFilePath}/{ConsolidatedMappingFileName}"
        
        # Load DataModel JSON
        temp_data_model = spark.read.format("json").option("multiline", "true").load(DataModelFile)
        data_model = temp_data_model.select(explode(col("Fields"))).select(
            col("col.FieldName").alias("FieldName"),
            col("col.DataType").alias("DataType"),
            col("col.Ordinal").alias("Ordinal")
        )
        
        # Create destination schema
        dest_schema = get_struct(data_model)
        df_data_model = spark.createDataFrame([], dest_schema)
        
        # Load ConsolidationMapping
        consolidated_mappings = spark.read.format("json").option("multiline", "true").load(ConsolidationMapping)
        temp_mappings = consolidated_mappings.select(explode(col("columnMapping"))).select(
            col("col.recordType").alias("recordType"),
            col("col.selectColumns").alias("selectColumns")
        )
        
        # Explode RecordType and selectColumns
        s_record_type = temp_mappings.select(explode(col("recordType"))).select(
            col("col.Field").alias("Field"),
            col("col.Value").alias("Value")
        )
        
        s_columns = temp_mappings.select(explode(col("selectColumns"))).select(
            col("col.SourceColumn").alias("SourceColumn"),
            col("col.DestinationColumn").alias("DestinationColumn"),
            col("col.SourceColumnFormat").alias("SourceColumnFormat"),
            col("col.ColumnQuery").alias("ColumnQuery")
        )
        
        # Merge mapping columns to datamodel
        seq_columns = data_model.join(
            s_columns,
            data_model["FieldName"] == s_columns["DestinationColumn"],
            "inner"
        ).select(
            data_model["FieldName"],
            data_model["DataType"],
            data_model["Ordinal"],
            s_columns["SourceColumn"],
            s_columns["DestinationColumn"],
            s_columns["SourceColumnFormat"],
            s_columns["ColumnQuery"]
        )
        
        # Get select expressions
        sel_cols = get_select_expr(seq_columns)
        
        # Create filter expression
        filter_rows = s_record_type.collect()
        if filter_rows:
            new_type = filter_rows[-1]
            Field = new_type.Field if new_type.Field else "FileId"
            Value = new_type.Value if new_type.Value else FileId
        else:
            Field = "FileId"
            Value = FileId
        
        # Load and transform data (from table or parquet file)
        if is_input_table:
            df_file_reformatted = spark.read.table(FullProcessed) \
                .selectExpr(*sel_cols) \
                .filter(col("FileID") == FileId) \
                .filter(col(Field) == Value)
        else:
            df_file_reformatted = spark.read.format("parquet").load(FullProcessed) \
                .selectExpr(*sel_cols) \
                .filter(col("FileID") == FileId) \
                .filter(col(Field) == Value)
        
        # Union with data model to enforce schema
        df_file = df_data_model.union(
            df_file_reformatted.select(custom_select(
                df_file_reformatted.columns,
                df_data_model.columns
            ))
        )
        
        # Write to delta (Serverless compatible - no .rdd access)
        # Check if DataFrame has rows using limit().count() instead of rdd.isEmpty()
        if df_file.limit(1).count() > 0:
            # Check if output is a table name or file path
            if FullConsolidatedFolderPath.count('.') == 2 and not FullConsolidatedFolderPath.startswith('/'):
                # It's a table name - use saveAsTable
                df_file.write.format("delta").option("mergeSchema", "true").mode("append").saveAsTable(FullConsolidatedFolderPath)
            else:
                # It's a file path - use save
                df_file.write.format("delta").option("mergeSchema", "true").mode("append").save(FullConsolidatedFolderPath)
        
        rJSON["ConsolidatedCount"] = str(df_file.count())
        rJSON["Status"] = "SUCCESS"
        rJSON["ErrorMessage"] = ""
        
    except Exception as e:
        rJSON["ConsolidatedCount"] = "0"
        rJSON["Status"] = "FAILED"
        rJSON["ErrorMessage"] = str(e).replace('"', '')
    
    return json.dumps(rJSON)

print("Helper functions and process_consolidation loaded")

In [0]:
rJSON_list = []

# Collect data rows to loop securely using native DataFrame actions
print("\nStep 3: Collecting rows for processing...")
rows = exploded_file_ids.collect()
print(f"Collected {len(rows)} row(s)")

In [0]:
print(f"\nStep 4: Processing each file...")
for idx, t in enumerate(rows, 1):
    print(f"\n--- Processing file {idx}/{len(rows)}: FileId={t['FileId']} ---")
    results = ""
    try:
        # Call process_consolidation function directly (no separate execution, no concurrency limit)
        print(f"  Calling process_consolidation function...")
        results = process_consolidation(
            FileId=str(t["FileId"]),
            CurrentContainer=str(t["CurrentContainer"]),
            CurrentFolderPath=str(t["CurrentFolderPath"]),
            ConsolidatedLayerDataModel=str(t["ConsolidatedLayerDataModel"]),
            ConsolidatedLayerDataModelFilePath=str(t["ConsolidatedLayerDataModelFilePath"]),
            ConsolidatedMappingFileName=str(t["ConsolidatedMappingFileName"]),
            ConsolidatedMappingFilePath=str(t["ConsolidatedMappingFilePath"]),
            ConsolidatedFolderPath=str(t["ConsolidatedFolderPath"])
        )
        print(f"  process_consolidation completed")
        print(f"  Raw result (first 200 chars): {results[:10]}...")
    except Exception as e:
        print(f"  process_consolidation failed: {str(e)}")
        results = json.dumps({
            "CurrentJobId": "None",
            "ConsolidatedCount": "0",
            "Status": "FAILED",
            "ErrorMessage": str(e)
        })

    # Process output metrics
    try:
        res_data = json.loads(results)
        print(f"  Result: {res_data.get('Status', 'UNKNOWN')} - {res_data.get('ConsolidatedCount', '0')} rows")
    except Exception as parse_err:
        res_data = {"CurrentJobId": "Error", "ConsolidatedCount": "0", "Status": "FAILED", "ErrorMessage": f"Failed to parse JSON: {str(parse_err)}. Raw result: {results[:10]}"}
        print(f"  Failed to parse result JSON: {str(parse_err)}")

    record = {
        "FileID": t["FileId"],
        "DataGroupTrackingID": t["DataGroupTrackingID"],
        "DataGroupMappingId": t["DataGroupMappingId"],
        "CurrentContainer": t["CurrentContainer"],
        "CurrentFolderPath": t["CurrentFolderPath"],
        "ConsolidatedFolderPath": t["ConsolidatedFolderPath"],
        "CurrentJobId": res_data.get("CurrentJobId", ""),
        "ConsolidatedCount": str(res_data.get("ConsolidatedCount", "0")),
        "Status": res_data.get("Status", ""),
        "ErrorMessage": res_data.get("ErrorMessage", "")
    }
    rJSON_list.append(record)

    print(f"\n All files processed. Total: {len(rJSON_list)}")

# Return the aggregate JSON string output back
result_json = json.dumps(rJSON_list)
print(f"\nReturning result JSON (length: {len(result_json)} chars)")
dbutils.notebook.exit(result_json)